# Phase 8 - Quick Pipeline Benchmark

This notebook runs the isolated stratified smoke-test pipeline. Quick-run metrics verify execution and performance only; they must not replace full-data research results.

In [ ]:
# Cell 1 - Locate the project root and import notebook dependencies.
from pathlib import Path
import json
import sys

import pandas as pd

project_root = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / 'configs' / 'config.yaml').exists()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

config_path = project_root / 'configs' / 'config.yaml'
print(f'Project root: {project_root}')

In [ ]:
# Cell 2 - Inspect the configured quick-run limits and isolated workspace.
from src.data_loading import load_config, resolve_project_path

config = load_config(config_path)
quick_config = config['pipeline']['quick_run']
quick_root = resolve_project_path(quick_config['root_dir'], project_root)

pd.Series(quick_config, name='configured_value').to_frame()

In [ ]:
# Cell 3 - Run or reuse the isolated all-scenario quick pipeline.
from run_pipeline import configure_logging
from src.quick_run import run_quick_pipeline

configure_logging('INFO')
quick_report = run_quick_pipeline(
    config_path=config_path,
    scenario='all',
    force=False,
)

print(f"Quick-run report: {quick_report['report_path']}")

In [ ]:
# Cell 4 - Summarize sampling reduction and measured execution time.
performance_fields = [
    'train_rows',
    'test_rows',
    'train_row_reduction_percentage',
    'test_row_reduction_percentage',
    'sample_preparation_seconds',
    'pipeline_seconds',
    'total_seconds',
]
pd.Series({field: quick_report[field] for field in performance_fields}, name='value').to_frame()

In [ ]:
# Cell 5 - Compare recorded cold-rebuild and cached quick-run timings.
quick_history = pd.read_csv(quick_report['history_path'])
quick_history.tail(10)

In [ ]:
# Cell 6 - Verify class coverage and inspect the sampled split distributions.
distribution_path = quick_root / 'results' / 'metrics' / 'split_class_distribution.csv'
quick_distribution = pd.read_csv(distribution_path)
assert quick_distribution.groupby('split')['class_index'].nunique().eq(10).all()
assert (quick_distribution['sample_count'] > 0).all()
quick_distribution

In [ ]:
# Cell 7 - Display status and duration for each phase in the quick pipeline.
with open(quick_report['pipeline_report_path'], 'r', encoding='utf-8') as file:
    pipeline_report = json.load(file)

phase_timing = pd.DataFrame(pipeline_report['phases'])
phase_timing

In [ ]:
# Cell 8 - Confirm isolation, source integrity, and smoke-test-only labeling.
full_processed_dir = resolve_project_path(config['paths']['processed_dir'], project_root)
quick_processed_dir = quick_root / 'processed'

assert quick_report['source_artifacts_unchanged'] is True
assert quick_report['scientific_use'] == 'smoke_test_only_not_final_results'
assert quick_report['train_row_reduction_percentage'] > 90.0
assert quick_report['test_row_reduction_percentage'] > 90.0
assert quick_processed_dir.resolve() != full_processed_dir.resolve()
assert phase_timing['phase'].tolist() == list(range(1, 8))

print('Quick-run isolation and integrity checks passed.')